# nn_ft_die_hpo — FT-Transformer **die-level** + Optuna HPO + HybridScaler + 결정성 모드

**목적**: 기존 unit-level FT-Transformer (val 0.005822, pred std 0.000033 — **거의 평균만 출력**) 의 학습 실패 원인 (seq_len 3,409 + unit 26K 표본) 을 die-level broadcast 학습으로 해결 시도.

**핵심 변경 (vs nn_ft_transformer_hpo)**:
- **die-level 학습**: 105K die row × ~568 raw feature (aggregate 안 함) → seq_len **3,409 → 569 (6x 단축)**
- **broadcast y**: `y_die = y_unit_broadcast` (4 die per unit 모두 동일 unit y)
- **CV split = ufs_serial 단위** (leakage 방지 — 같은 unit 의 4 die 가 train/val 에 섞이면 안 됨)
- **inference**: predict per-die → mean 4 die → unit pred (03b/03f 와 동일 패턴)

**unit-level 노트북에서 이식한 개선사항**:
- **HybridScaler**: binary passthrough + |skew|>10 → Quantile + 나머지 → Power(Yeo-Johnson). 분포 왜곡 강한 feature 에 robust
- **DETERMINISTIC 모드**: cudnn deterministic + `use_deterministic_algorithms` + `CUBLAS_WORKSPACE_CONFIG` (재현성, ~5~15% 속도 손실)
- **RUN_HPO 토글**: True 면 HPO + refit, False 면 `BEST_HP_FROM_DB` 로 refit 만 (빠른 비교)
- **SAVE_FOLD_MODELS**: fold별 best epoch state_dict `.pt` 저장 (재현/추론 가능)
- **scaler.pkl** 저장 + **ENV_INFO** (torch/cuda/cudnn version) meta 기록

**진단 목표**:
1. NN val pred std > 0.001 → 평균 예측 탈출 성공 (signal 학습)
2. val RMSE < 0.0058 → plateau 안 → stacking 후보
3. 둘 다 실패 → NN architecture 자체의 한계 확정

**격리**: `4_output/_temp/nn_ft_die_hybrid/` 신규.

**비교 기준**:
- 기존 nn_ft (unit-level + StandardScaler): val=0.005822 ← 평균 예측에 갇힘
- 03b (die-level LGBM Two-Stage): val=0.005718, test=0.008417
- zit_only: val=0.005709

## 1. 환경 + import (Colab GPU / Local 공통)

In [1]:
import os, sys, json, math, random, pickle

# === 결정성 (torch import 전에 환경변수 설정) ===
DETERMINISTIC = True   # True면 cudnn deterministic + 알고리즘 고정. 약 5~15% 속도 손실
if DETERMINISTIC:
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

GDRIVE_FINAL_ID = '1HR7LlQmp4n9wGh2WneyVex2mCZ-poiY9'

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system('gdown 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/final/modules/preprocess.py'):
        assert GDRIVE_FINAL_ID, 'GDRIVE_FINAL_ID 비어있음'
        os.makedirs('/content/project/3_modeling/final', exist_ok=True)
        os.system(f'gdown {GDRIVE_FINAL_ID} -O /content/final.zip')
        os.system('unzip -qo /content/final.zip -d /content/project/3_modeling/final')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext

import optuna
from sklearn.model_selection import KFold

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)
from final.modules import preprocess
from final.modules.scaling import HybridScaler

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision('high')   # TF32 matmul (정밀도 손실 미미)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    if DETERMINISTIC:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception as _e:
            print(f'[deterministic] use_deterministic_algorithms 설정 실패: {_e}')
    else:
        torch.backends.cudnn.benchmark = True

USE_AMP            = (DEVICE == 'cuda')
USE_COMPILE_HPO    = False
USE_COMPILE_REFIT  = (DEVICE == 'cuda') and (not DETERMINISTIC)  # compile은 비결정성 유발 가능

optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'DEVICE = {DEVICE}')
print(f'PyTorch = {torch.__version__}, CUDA available = {torch.cuda.is_available()}')
if DEVICE == 'cuda':
    print(f'GPU = {torch.cuda.get_device_name(0)}')
    print(f'AMP(bf16)={USE_AMP}, compile(HPO/refit)={USE_COMPILE_HPO}/{USE_COMPILE_REFIT}, '
          f'TF32=ON, deterministic={DETERMINISTIC}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
setup 완료
PROJECT_ROOT = /content/project
DEVICE = cuda
PyTorch = 2.10.0+cu128, CUDA available = True
GPU = NVIDIA A100-SXM4-80GB
AMP(bf16)=True, compile(HPO/refit)=False/False, TF32=ON, deterministic=True


In [2]:
# === 인라인 핫픽스: HybridScaler.fit — Yeo-Johnson 컬럼별 fallback ===
# 사유: PowerTransformer(yeo-johnson)의 Brent 옵티마이저가 일부 die-level 컬럼에서
#       valid bracket을 못 찾고 BracketError 로 깨짐 (단조 NLL / 극단값 문제).
# 처치: power_cols 를 컬럼별로 fit 시도 → 실패한 컬럼만 Quantile 그룹으로 자동 이관.
#       원본 scaling.py 는 그대로 두고, 본 셀에서만 메서드 교체.
# 정식 패치 후 (final.zip 재업로드) 본 셀 제거 권장.

from final.modules.scaling import HybridScaler
from sklearn.preprocessing import PowerTransformer, QuantileTransformer
import pandas as _pd_patch

_orig_hybrid_fit = HybridScaler.fit  # 보존 (필요 시 HybridScaler.fit = _orig_hybrid_fit 로 복원)

def _patched_hybrid_fit(self, X, feat_cols=None):
    if feat_cols is None:
        feat_cols = list(X.columns)
    self.feat_cols_ = list(feat_cols)

    # 1) Binary passthrough (nunique ≤ 2)
    if self.binary_passthrough:
        nuniq = X[self.feat_cols_].nunique()
        self.binary_cols_ = nuniq[nuniq <= 2].index.tolist()
    else:
        self.binary_cols_ = []
    remaining = [c for c in self.feat_cols_ if c not in set(self.binary_cols_)]

    # 2) Skew 기준 분기
    if remaining:
        skew_vals = X[remaining].skew().abs()
    else:
        skew_vals = _pd_patch.Series(dtype=float)
    self.skew_vals_ = skew_vals
    self.quantile_cols_ = skew_vals[skew_vals > self.skew_threshold].index.tolist()
    self.power_cols_ = [c for c in remaining if c not in set(self.quantile_cols_)]

    # 3) Power 그룹 — 컬럼별 try/except. 실패 컬럼은 quantile 로 이관
    self.pt_ = None
    pt_failed = []  # (col, exc_type)
    if self.power_cols_:
        ok_cols = []
        for c in self.power_cols_:
            try:
                _tmp = PowerTransformer(method='yeo-johnson', standardize=True)
                _tmp.fit(X[[c]])
                ok_cols.append(c)
            except Exception as _e:
                pt_failed.append((c, type(_e).__name__))
        if ok_cols:
            self.pt_ = PowerTransformer(method='yeo-johnson', standardize=True)
            self.pt_.fit(X[ok_cols])
        if pt_failed:
            failed_only = [c for c, _ in pt_failed]
            self.quantile_cols_ = list(self.quantile_cols_) + failed_only
        self.power_cols_ = ok_cols
    self._pt_failed_cols_ = pt_failed  # 진단용

    # 4) Quantile fit (확장된 quantile_cols_ 기준, train 길이로 n_quantiles 클램프)
    self.qt_ = None
    if self.quantile_cols_:
        n_q = min(self.n_quantiles, len(X))
        self.qt_ = QuantileTransformer(
            n_quantiles=n_q,
            output_distribution=self.quantile_output,
            subsample=int(1e6),
            random_state=self.random_state,
        )
        self.qt_.fit(X[self.quantile_cols_])

    print(f"[HybridScaler.fit/PATCH] skew_threshold={self.skew_threshold}")
    if self.binary_passthrough:
        print(f"  Binary passthrough: {len(self.binary_cols_)}개 (nunique ≤ 2, 변환 없음)")
    print(f"  Quantile 적용     : {len(self.quantile_cols_)}개 "
          f"(|skew|>{self.skew_threshold} + Yeo-Johnson 실패 fallback {len(pt_failed)}개)")
    print(f"  Power 적용        : {len(self.power_cols_)}개 (Yeo-Johnson + standardize)")
    if pt_failed:
        print(f"  [PATCH] Yeo-Johnson 실패 → Quantile 로 이관된 컬럼:")
        for c, et in pt_failed[:10]:
            print(f"    - {c}  ({et})")
        if len(pt_failed) > 10:
            print(f"    ... +{len(pt_failed) - 10}개 추가")
    return self

HybridScaler.fit = _patched_hybrid_fit
print('[PATCH] HybridScaler.fit 인라인 핫픽스 적용 완료 '
      '(Yeo-Johnson 실패 컬럼 → Quantile fallback)')

[PATCH] HybridScaler.fit 인라인 핫픽스 적용 완료 (Yeo-Johnson 실패 컬럼 → Quantile fallback)


## 2. 설정 (die-level)

In [3]:
EXP_ID   = 'nn-ft-die-002-hybrid'
EXP_MEMO = 'FT-Transformer die-level + HybridScaler + RUN_HPO toggle + 결정성 모드 + fold state save'
USER     = 'jh'

# === HPO 토글 ===
# True  : 30 trial Optuna HPO + 5-fold refit (full)
# False : HPO 건너뛰고 BEST_HP_FROM_DB로 5-fold refit만 (빠름, 시나리오 비교용)
RUN_HPO = True

# === 기존 unit-level best HP (placeholder — die-level HPO 후 갱신 권장) ===
# RUN_HPO=False 일 때만 사용. die-level 적정 HP 가 unit-level과 다를 수 있으므로
# 첫 die-level HPO 끝나면 그 결과로 교체하는 게 정석.
BEST_HP_FROM_DB = {
    'd_model':      64,
    'n_layers':     4,
    'dropout':      0.2829012260843204,
    'lr':           0.0024939396315541013,
    'weight_decay': 1.0345436422454988e-05,
    'batch_size':   2048,
    'loss_type':    'mse',
    'n_heads':      4,    # fixed
    'ffn_factor':   2,    # fixed
}

N_TRIALS       = 30
N_FOLDS_HPO    = 3
N_FOLDS_REFIT  = 5
MAX_EPOCHS     = 30     # die-level은 4x 샘플이라 epoch 적게 (그래도 update 수 충분)
PATIENCE       = 5
CLIP_Y_EXTREME = True
SAVE_FOLD_MODELS = True   # fold별 state_dict .pt 저장 (재현성)

# OUT_DIR — die-level + Hybrid 격리
OUT_DIR = os.path.join(OUTPUT_DIR, '_temp', 'nn_ft_die_hybrid')
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# 03b log1p preset (검증된 PP)
PARAMS = {
    'missing_threshold':          0.5,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.25,
    'spatial_max_dist':           5.0,
    'post_impute_corr_threshold': 0.99,
    'post_impute_corr_keep_by':   'std',
}

print(f'EXP_ID={EXP_ID}')
print(f'RUN_HPO={RUN_HPO}  (False면 BEST_HP_FROM_DB로 refit만)')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS_HPO={N_FOLDS_HPO} | N_FOLDS_REFIT={N_FOLDS_REFIT}')
print(f'MAX_EPOCHS={MAX_EPOCHS} | PATIENCE={PATIENCE} | CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'SAVE_FOLD_MODELS={SAVE_FOLD_MODELS}')
print(f'OUT_DIR={OUT_DIR}')

EXP_ID=nn-ft-die-002-hybrid
RUN_HPO=True  (False면 BEST_HP_FROM_DB로 refit만)
N_TRIALS=30 | N_FOLDS_HPO=3 | N_FOLDS_REFIT=5
MAX_EPOCHS=30 | PATIENCE=5 | CLIP_Y_EXTREME=True
SAVE_FOLD_MODELS=True
OUT_DIR=/content/project/4_output/_temp/nn_ft_die_hybrid


## 3. 데이터 로드 + Y clip + 전처리 + die-level 변환 (no aggregate)

- die-level cleaning → ~568 features
- y_die_broadcast: 각 die row 가 자신의 unit y 를 받음 (4 die per unit 모두 동일)
- StandardScaler: train die row 기준 fit → all transform
- uid_*_die: 각 die row 의 ufs_serial (mean aggregation 시 unit grouping 용)

In [4]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

# ── die-level cleaning ──
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']
print(f'\n[cleaning] feat_cols_clean={len(feat_cols_clean)}')

# ── die-level X (DataFrame, no aggregate!) ──
X_train_die_df = xs_train_die[feat_cols_clean].copy()
X_val_die_df   = xs_val_die  [feat_cols_clean].copy()
X_test_die_df  = xs_test_die [feat_cols_clean].copy()

# ── HybridScaler (binary passthrough + |skew|>10 → Quantile + 나머지 → Power) ──
SCALER_KW = dict(
    skew_threshold=10.0,
    n_quantiles=1000,
    quantile_output='normal',
    random_state=SEED,
    binary_passthrough=True,
)
scaler = HybridScaler(**SCALER_KW).fit(X_train_die_df)
X_train_die_s_df = scaler.transform(X_train_die_df, inplace=False)
X_val_die_s_df   = scaler.transform(X_val_die_df,   inplace=False)
X_test_die_s_df  = scaler.transform(X_test_die_df,  inplace=False)

X_train_die_s = X_train_die_s_df.values.astype(np.float32)
X_val_die_s   = X_val_die_s_df.values.astype(np.float32)
X_test_die_s  = X_test_die_s_df.values.astype(np.float32)

uid_train_die = xs_train_die[KEY_COL].values
uid_val_die   = xs_val_die  [KEY_COL].values
uid_test_die  = xs_test_die [KEY_COL].values

# ── broadcast y to die rows ──
y_train_die_broadcast = pd.Series(uid_train_die).map(y_train_unit).values.astype(np.float32)
assert not pd.isna(y_train_die_broadcast).any(), 'unmapped train die y'

N_FEATURES = X_train_die_s.shape[1]
SCALER_STATS = {
    'type':              'HybridScaler',
    'skew_threshold':    SCALER_KW['skew_threshold'],
    'n_quantiles':       SCALER_KW['n_quantiles'],
    'quantile_output':   SCALER_KW['quantile_output'],
    'binary_passthrough': SCALER_KW['binary_passthrough'],
    'binary_n':          len(scaler.binary_cols_),
    'quantile_n':        len(scaler.quantile_cols_),
    'power_n':           len(scaler.power_cols_),
}
print(f'\n[die-level + HybridScale] N_FEATURES={N_FEATURES} (vs unit-level 3,408 → 6x 단축)')
print(f'  X_train_die_s: {X_train_die_s.shape} (105K rows vs unit 26K — 4x 샘플)')
print(f'  X_val_die_s:   {X_val_die_s.shape}')
print(f'  X_test_die_s:  {X_test_die_s.shape}')
print(f'  scaler 분배: binary={SCALER_STATS["binary_n"]}, '
      f'quantile={SCALER_STATS["quantile_n"]}, power={SCALER_STATS["power_n"]}')
print(f'  y_train_die mean = {y_train_die_broadcast.mean():.6f}, '
      f'std = {y_train_die_broadcast.std():.6f}, '
      f'pos ratio = {(y_train_die_broadcast > 0).mean():.4f}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=50%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 332개, 잔여: 564개
    컬럼: 896 → 564 (332개 제거)
    DataFrame: (104748, 622)

[결측 indicator] 4개 컬럼 추가 (결측률 >= 25%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, dist<=5

## 4. FT-Transformer 모델 + Loss 정의

**FT-Transformer** (Yandex 2021): NumericalTokenizer + CLS + N×TransformerBlock → MLP head. 이전 노트북과 동일.

**Loss**:
- MSE: `F.mse_loss(pred_log, log1p(y_die_broadcast))` — die-level loss
- Tweedie: same as before

die-level 학습이라도 loss 자체는 row-level (die row × broadcast y), 동일.

In [5]:
class NumericalTokenizer(nn.Module):
    def __init__(self, n_features, d_model):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_features, d_model))
        self.bias   = nn.Parameter(torch.zeros(n_features, d_model))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
    def forward(self, x):
        return x.unsqueeze(-1) * self.weight + self.bias


class FTBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout, ffn_factor):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_model * ffn_factor),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * ffn_factor, d_model),
        )
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        h = self.norm1(x)
        a, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.dropout(a)
        h = self.norm2(x)
        x = x + self.dropout(self.ffn(h))
        return x


class FTTransformer(nn.Module):
    def __init__(self, n_features, d_model=64, n_heads=4, n_layers=3,
                 dropout=0.1, ffn_factor=2):
        super().__init__()
        self.tokenizer = NumericalTokenizer(n_features, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls_token, std=0.02)
        self.blocks = nn.ModuleList([
            FTBlock(d_model, n_heads, dropout, ffn_factor)
            for _ in range(n_layers)
        ])
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1),
        )
    def forward(self, x):
        h = self.tokenizer(x)
        cls = self.cls_token.expand(h.size(0), -1, -1)
        h = torch.cat([cls, h], dim=1)
        for blk in self.blocks:
            h = blk(h)
        return self.head(h[:, 0]).squeeze(-1)


def loss_mse(pred_log, y_orig, **kwargs):
    target_log = torch.log1p(y_orig)
    return F.mse_loss(pred_log, target_log)


def loss_tweedie(pred_log, y_orig, power=1.5, **kwargs):
    mu = torch.clamp(torch.expm1(pred_log), min=1e-6)
    a = y_orig * (mu ** (1 - power)) / (1 - power)
    b = (mu ** (2 - power)) / (2 - power)
    return torch.mean(b - a)


def get_loss_fn(loss_type):
    if loss_type == 'mse':
        return loss_mse
    elif loss_type.startswith('tweedie'):
        power = float(loss_type.split('_')[1])
        return lambda pred_log, y_orig: loss_tweedie(pred_log, y_orig, power=power)
    raise ValueError(f'unknown loss: {loss_type}')


def predict_y(pred_log_np):
    return np.clip(np.expm1(pred_log_np), 0.0, None)


print('FT-Transformer + Loss 정의 완료 (die-level — broadcast y, mean agg later)')

FT-Transformer + Loss 정의 완료 (die-level — broadcast y, mean agg later)


## 5. die→unit 집계 헬퍼 + 1-fold 학습 함수

**핵심 차이 (vs unit-level)**: 학습은 die row 단위, **early stopping 평가/저장은 unit RMSE** (예측 4 die mean → unit pred → 비교).

In [6]:
def _rmse(pred, true):
    return float(np.sqrt(np.mean((np.asarray(pred) - np.asarray(true)) ** 2)))


def _amp_ctx():
    if USE_AMP:
        return torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16)
    return nullcontext()


def _aggregate_die_to_unit(pred_die, uid_die):
    """die predictions → unit predictions (mean of 4 dies per unit).
    Returns (unit_pred, unit_ids_sorted)."""
    uid_arr = np.asarray(uid_die)
    unique_units, inverse = np.unique(uid_arr, return_inverse=True)
    pred_sum = np.zeros(len(unique_units), dtype=np.float64)
    cnt      = np.zeros(len(unique_units), dtype=np.float64)
    np.add.at(pred_sum, inverse, pred_die)
    np.add.at(cnt,      inverse, 1.0)
    return (pred_sum / cnt).astype(np.float32), unique_units


def _predict_batched(model, X_t, bs=4096):
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, X_t.size(0), bs):
            with _amp_ctx():
                pred = model(X_t[i:i+bs])
            outs.append(pred.float().cpu().numpy())
    return np.concatenate(outs, axis=0)


def _train_one_fold_impl(X_tr, y_tr_die,                          # die-level train
                          X_vl, y_vl_unit_arr, uid_vl,             # die-level X, unit-level y, die uid
                          X_others=None, uid_others_list=None,     # die-level X 들 + uid 들
                          hp=None, max_epochs=MAX_EPOCHS, patience=PATIENCE,
                          verbose=False, use_compile=False):
    """die-level 학습 + unit RMSE 기반 early stopping.
    Returns dict with die-level + unit-level predictions.
    """
    model = FTTransformer(
        n_features=N_FEATURES,
        d_model=hp['d_model'], n_heads=hp['n_heads'],
        n_layers=hp['n_layers'], dropout=hp['dropout'],
        ffn_factor=hp['ffn_factor'],
    ).to(DEVICE)

    if use_compile:
        try:
            model = torch.compile(model, mode='default', dynamic=True)
        except Exception as _e:
            if verbose:
                print(f'    compile skipped: {_e}')

    opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['weight_decay'])
    loss_fn = get_loss_fn(hp['loss_type'])

    Xtr_t = torch.from_numpy(X_tr).to(DEVICE)
    ytr_t = torch.from_numpy(y_tr_die).to(DEVICE)
    Xvl_t = torch.from_numpy(X_vl).to(DEVICE)
    others_t = [torch.from_numpy(X).to(DEVICE) for X in (X_others or [])]

    n_train = X_tr.shape[0]
    bs = hp['batch_size']

    best_val_rmse = float('inf')
    best_oof_die = None
    best_oof_unit = None
    best_others_die = None
    best_others_unit = None
    bad_count = 0
    best_epoch = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        perm = torch.randperm(n_train, device=DEVICE)
        epoch_loss = 0.0
        for i in range(0, n_train, bs):
            idx = perm[i:i+bs]
            xb = Xtr_t[idx]
            yb = ytr_t[idx]
            opt.zero_grad(set_to_none=True)
            with _amp_ctx():
                pred_log = model(xb)
            loss = loss_fn(pred_log.float(), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            epoch_loss += loss.item() * xb.size(0)
        epoch_loss /= n_train

        # ── eval: predict per-die → mean to unit → RMSE vs unit y ──
        vl_pred_die_log = _predict_batched(model, Xvl_t, bs=bs * 4)
        vl_pred_die = predict_y(vl_pred_die_log)
        vl_pred_unit, _ = _aggregate_die_to_unit(vl_pred_die, uid_vl)
        vl_rmse = _rmse(vl_pred_unit, y_vl_unit_arr)

        if verbose and (epoch == 1 or epoch % 5 == 0):
            print(f'    ep{epoch:3d}  loss={epoch_loss:.6f}  val_unit_rmse={vl_rmse:.6f}')

        if vl_rmse < best_val_rmse - 1e-7:
            best_val_rmse = vl_rmse
            best_oof_die = vl_pred_die
            best_oof_unit = vl_pred_unit
            best_epoch = epoch
            best_others_die = []
            best_others_unit = []
            for X_o, uid_o in zip(others_t, uid_others_list or []):
                p_die = predict_y(_predict_batched(model, X_o, bs=bs * 4))
                p_unit, _ = _aggregate_die_to_unit(p_die, uid_o)
                best_others_die.append(p_die)
                best_others_unit.append(p_unit)
            bad_count = 0
        else:
            bad_count += 1
            if bad_count >= patience:
                break

    if best_oof_die is None:
        # fallback: last epoch
        vl_pred_die = predict_y(_predict_batched(model, Xvl_t, bs=bs * 4))
        vl_pred_unit, _ = _aggregate_die_to_unit(vl_pred_die, uid_vl)
        best_oof_die = vl_pred_die
        best_oof_unit = vl_pred_unit
        best_val_rmse = _rmse(vl_pred_unit, y_vl_unit_arr)
        best_others_die = []
        best_others_unit = []
        for X_o, uid_o in zip(others_t, uid_others_list or []):
            p_die = predict_y(_predict_batched(model, X_o, bs=bs * 4))
            p_unit, _ = _aggregate_die_to_unit(p_die, uid_o)
            best_others_die.append(p_die)
            best_others_unit.append(p_unit)

    del model, opt, Xtr_t, ytr_t, Xvl_t, others_t
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return {
        'oof_die':     best_oof_die,
        'oof_unit':    best_oof_unit,
        'others_die':  best_others_die,
        'others_unit': best_others_unit,
        'val_rmse':    best_val_rmse,
        'epoch':       best_epoch,
    }


def train_one_fold(X_tr, y_tr_die, X_vl, y_vl_unit_arr, uid_vl,
                    X_others=None, uid_others_list=None,
                    hp=None, max_epochs=MAX_EPOCHS, patience=PATIENCE,
                    verbose=False, min_bs=64, use_compile=False):
    """OOM fallback wrapper."""
    hp = dict(hp)
    orig_bs = hp['batch_size']
    while hp['batch_size'] >= min_bs:
        try:
            return _train_one_fold_impl(
                X_tr, y_tr_die, X_vl, y_vl_unit_arr, uid_vl,
                X_others=X_others, uid_others_list=uid_others_list,
                hp=hp, max_epochs=max_epochs, patience=patience,
                verbose=verbose, use_compile=use_compile,
            )
        except torch.cuda.OutOfMemoryError as _e:
            torch.cuda.empty_cache()
            new_bs = hp['batch_size'] // 2
            print(f'  [OOM] bs {hp["batch_size"]} → {new_bs} fallback')
            hp['batch_size'] = new_bs
    raise RuntimeError(f'OOM 지속 (bs<{min_bs})')


print('train_one_fold (die-level + unit eval) 정의 완료')

train_one_fold (die-level + unit eval) 정의 완료


## 6. Optuna HPO

**핵심**: KFold split 은 **ufs_serial 단위** (leakage 방지). die row 마스킹으로 train/val die 결정.

**objective metric**: 3-fold OOF unit RMSE (각 fold 의 val unit pred 누적 → 전체 RMSE).

**MedianPruner** + **compile OFF** (HPO 단계).

In [ ]:
import time

# unit-level fold split (leakage 방지)
unit_ids_train_unique = y_train_unit.index.values
kf_refit = KFold(n_splits=N_FOLDS_REFIT, shuffle=True, random_state=SEED)
FOLDS_REFIT = list(kf_refit.split(np.arange(len(unit_ids_train_unique))))

y_train_unit_dict = pd.Series(y_train_unit.values, index=y_train_unit.index)


def _build_fold_args(tr_uidx, vl_uidx):
    """unit-fold idx → die-level X/y/uid 짝."""
    tr_units = unit_ids_train_unique[tr_uidx]
    vl_units = unit_ids_train_unique[vl_uidx]
    tr_mask = np.isin(uid_train_die, tr_units)
    vl_mask = np.isin(uid_train_die, vl_units)
    X_tr = X_train_die_s[tr_mask]
    y_tr = y_train_die_broadcast[tr_mask]
    X_vl = X_train_die_s[vl_mask]
    uid_vl = uid_train_die[vl_mask]
    sorted_vl_units = np.unique(uid_vl)
    y_vl_unit_arr = y_train_unit_dict.loc[sorted_vl_units].values.astype(np.float32)
    return X_tr, y_tr, X_vl, y_vl_unit_arr, uid_vl, sorted_vl_units, vl_mask


study = None  # RUN_HPO=False 분기에서 cell-save가 참조

if RUN_HPO:
    kf_hpo = KFold(n_splits=N_FOLDS_HPO, shuffle=True, random_state=SEED)
    FOLDS_HPO = list(kf_hpo.split(np.arange(len(unit_ids_train_unique))))


    def objective(trial):
        hp = {
            'd_model':      trial.suggest_categorical('d_model', [32, 48, 64, 96]),
            'n_heads':      4,
            'n_layers':     trial.suggest_int('n_layers', 2, 4),
            'dropout':      trial.suggest_float('dropout', 0.05, 0.4),
            'ffn_factor':   2,
            'lr':           trial.suggest_float('lr', 1e-4, 5e-3, log=True),
            'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
            'batch_size':   trial.suggest_categorical('batch_size', [512, 1024, 2048, 4096]),
            'loss_type':    trial.suggest_categorical(
                'loss_type', ['mse', 'tweedie_1.2', 'tweedie_1.5', 'tweedie_1.7']
            ),
        }

        oof_unit_full = pd.Series(np.zeros(len(unit_ids_train_unique), dtype=np.float32),
                                  index=unit_ids_train_unique)
        fold_val_rmses = []
        t0 = time.time()
        for fi, (tr_uidx, vl_uidx) in enumerate(FOLDS_HPO):
            X_tr, y_tr, X_vl, y_vl_unit_arr, uid_vl, sorted_vl_units, _ = _build_fold_args(tr_uidx, vl_uidx)
            result = train_one_fold(
                X_tr, y_tr, X_vl, y_vl_unit_arr, uid_vl,
                X_others=None, uid_others_list=None,
                hp=hp, use_compile=USE_COMPILE_HPO,
            )
            oof_unit_full.loc[sorted_vl_units] = result['oof_unit']
            fold_val_rmses.append(result['val_rmse'])
            trial.report(result['val_rmse'], fi)
            if trial.should_prune():
                trial.set_user_attr('pruned_at_fold', fi)
                trial.set_user_attr('elapsed_s', time.time() - t0)
                raise optuna.TrialPruned()

        y_full = y_train_unit_dict.loc[unit_ids_train_unique].values
        oof_rmse_score = _rmse(oof_unit_full.values, y_full)
        trial.set_user_attr('oof_rmse', oof_rmse_score)
        trial.set_user_attr('mean_fold_val', float(np.mean(fold_val_rmses)))
        trial.set_user_attr('elapsed_s', time.time() - t0)
        trial.set_user_attr('pred_std', float(oof_unit_full.std()))
        return oof_rmse_score


    study = optuna.create_study(
        direction='minimize',
        study_name=EXP_ID,
        storage=f'sqlite:///{DB_PATH}',
        load_if_exists=False,
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0),
    )
    study.set_user_attr('exp_memo', EXP_MEMO)
    study.set_user_attr('device', DEVICE)
    study.set_user_attr('training_level', 'die')
    study.set_user_attr('n_features', int(N_FEATURES))
    study.set_user_attr('n_train_die', int(X_train_die_s.shape[0]))
    study.set_user_attr('scaler', 'HybridScaler')
    study.set_user_attr('deterministic', DETERMINISTIC)
    study.set_user_attr('compile_hpo', USE_COMPILE_HPO)
    study.set_user_attr('compile_refit', USE_COMPILE_REFIT)

    print(f'=== Optuna HPO 시작 (die-level, N_TRIALS={N_TRIALS}) ===')
    print(f'    train die rows : {X_train_die_s.shape[0]:,}')
    print(f'    N_FEATURES     : {N_FEATURES}')
    print(f'    seq_len (model): {N_FEATURES + 1} (CLS + features)')
    print(f'    batch_size grid: [512, 1024, 2048, 4096]')
    print(f'    scaler         : HybridScaler (binary={SCALER_STATS["binary_n"]}, '
          f'quantile={SCALER_STATS["quantile_n"]}, power={SCALER_STATS["power_n"]})')
    print(f'    deterministic  : {DETERMINISTIC}')
    t_total = time.time()
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    print(f'\n[HPO 완료] {time.time()-t_total:.0f}s')
    n_pruned   = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
    n_complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    print(f'  trials: complete={n_complete}, pruned={n_pruned}')
    print(f'  best OOF unit RMSE = {study.best_value:.6f}')
    print(f'  best params:')
    for k, v in study.best_trial.params.items():
        print(f'    {k:15s} = {v}')
    print(f'  best pred_std = {study.best_trial.user_attrs.get("pred_std"):.6f}  (>0.001 면 평균 예측 탈출)')
else:
    print('=== RUN_HPO=False → HPO 건너뜀. BEST_HP_FROM_DB로 바로 refit ===')
    print(f'  BEST_HP_FROM_DB:')
    for k, v in BEST_HP_FROM_DB.items():
        print(f'    {k:15s} = {v}')

=== Optuna HPO 시작 (die-level, N_TRIALS=30) ===
    train die rows : 104,748
    N_FEATURES     : 568
    seq_len (model): 569 (CLS + features)
    batch_size grid: [512, 1024, 2048, 4096]
    scaler         : HybridScaler (binary=32, quantile=42, power=494)
    deterministic  : True


  0%|          | 0/30 [00:00<?, ?it/s]

## 7. Best params 5-fold refit + OOF/val/test (die + unit)

In [ ]:
# best_hp 결정: HPO 결과 우선, 없으면 DB 하드코딩 사용
if RUN_HPO and study is not None:
    best_hp = dict(study.best_trial.params)
    best_hp['n_heads']    = 4
    best_hp['ffn_factor'] = 2
    HP_SOURCE = 'optuna_current_run'
else:
    best_hp = dict(BEST_HP_FROM_DB)
    HP_SOURCE = 'fixed_from_BEST_HP_FROM_DB (placeholder unit-level)'

print(f'best_hp source: {HP_SOURCE}')

# accumulators (die-level)
oof_die_arr  = np.zeros(len(uid_train_die),  dtype=np.float32)
oof_die_mask = np.zeros(len(uid_train_die),  dtype=bool)
val_die_arr  = np.zeros(len(uid_val_die),    dtype=np.float32)
test_die_arr = np.zeros(len(uid_test_die),   dtype=np.float32)
fold_records = []
fold_state_paths = []

print(f'=== Refit (best params) {N_FOLDS_REFIT}-fold '
      f'(compile={USE_COMPILE_REFIT}, deterministic={DETERMINISTIC}, '
      f'save_states={SAVE_FOLD_MODELS}) ===')
t0 = time.time()

if SAVE_FOLD_MODELS:
    # ── 직접 학습 루프 (die-level + best epoch state save) ──
    fold_state_dir = os.path.join(OUT_DIR, 'fold_states')
    os.makedirs(fold_state_dir, exist_ok=True)

    Xval_t  = torch.from_numpy(X_val_die_s).to(DEVICE)
    Xtest_t = torch.from_numpy(X_test_die_s).to(DEVICE)

    for fi, (tr_uidx, vl_uidx) in enumerate(FOLDS_REFIT):
        # fold별 시드 고정
        torch.manual_seed(SEED + fi)
        if DEVICE == 'cuda':
            torch.cuda.manual_seed_all(SEED + fi)

        X_tr, y_tr_die, X_vl, y_vl_unit_arr, uid_vl, sorted_vl_units, vl_die_mask = _build_fold_args(tr_uidx, vl_uidx)

        model = FTTransformer(
            n_features=N_FEATURES,
            d_model=best_hp['d_model'],
            n_heads=best_hp['n_heads'],
            n_layers=best_hp['n_layers'],
            dropout=best_hp['dropout'],
            ffn_factor=best_hp['ffn_factor'],
        ).to(DEVICE)
        opt = torch.optim.AdamW(
            model.parameters(),
            lr=best_hp['lr'],
            weight_decay=best_hp['weight_decay'],
        )
        loss_fn = get_loss_fn(best_hp['loss_type'])

        Xtr_t = torch.from_numpy(X_tr).to(DEVICE)
        ytr_t = torch.from_numpy(y_tr_die).to(DEVICE)
        Xvl_t = torch.from_numpy(X_vl).to(DEVICE)

        n_train_fold = Xtr_t.shape[0]
        bs_fold = best_hp['batch_size']
        best_vl = float('inf')
        best_state = None
        best_oof_die_pred = None
        best_val_die_pred = None
        best_test_die_pred = None
        best_epoch = 0
        bad = 0

        for ep in range(1, MAX_EPOCHS + 1):
            model.train()
            perm = torch.randperm(n_train_fold, device=DEVICE)
            epoch_loss = 0.0
            for i in range(0, n_train_fold, bs_fold):
                idx = perm[i:i+bs_fold]
                opt.zero_grad(set_to_none=True)
                with _amp_ctx():
                    pred_log = model(Xtr_t[idx])
                loss = loss_fn(pred_log.float(), ytr_t[idx])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                epoch_loss += loss.item() * idx.size(0)
            epoch_loss /= n_train_fold

            # eval — die predict → unit aggregate → unit RMSE
            vl_pred_die_log = _predict_batched(model, Xvl_t, bs=bs_fold * 4)
            vl_pred_die = predict_y(vl_pred_die_log)
            vl_pred_unit, _ = _aggregate_die_to_unit(vl_pred_die, uid_vl)
            vl_rmse_ep = _rmse(vl_pred_unit, y_vl_unit_arr)

            if (fi == 0) and (ep == 1 or ep % 5 == 0):
                print(f'    ep{ep:3d}  loss={epoch_loss:.6f}  val_unit_rmse={vl_rmse_ep:.6f}')

            if vl_rmse_ep < best_vl - 1e-7:
                best_vl = vl_rmse_ep
                best_oof_die_pred = vl_pred_die
                best_val_die_pred = predict_y(_predict_batched(model, Xval_t,  bs=bs_fold * 4))
                best_test_die_pred = predict_y(_predict_batched(model, Xtest_t, bs=bs_fold * 4))
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_epoch = ep
                bad = 0
            else:
                bad += 1
                if bad >= PATIENCE:
                    break

        if best_oof_die_pred is None:
            # fallback: 마지막 epoch 결과
            best_oof_die_pred = predict_y(_predict_batched(model, Xvl_t,  bs=bs_fold * 4))
            best_val_die_pred = predict_y(_predict_batched(model, Xval_t, bs=bs_fold * 4))
            best_test_die_pred = predict_y(_predict_batched(model, Xtest_t, bs=bs_fold * 4))
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            vl_pred_unit, _ = _aggregate_die_to_unit(best_oof_die_pred, uid_vl)
            best_vl = _rmse(vl_pred_unit, y_vl_unit_arr)

        # die-level 누적
        oof_die_arr[vl_die_mask]  = best_oof_die_pred
        oof_die_mask[vl_die_mask] = True
        val_die_arr  += best_val_die_pred  / N_FOLDS_REFIT
        test_die_arr += best_test_die_pred / N_FOLDS_REFIT
        fold_records.append({'fold': fi+1, 'val_rmse': best_vl, 'epochs': best_epoch})

        # state_dict 저장
        state_path = os.path.join(fold_state_dir, f'fold_{fi+1}.pt')
        torch.save({
            'state_dict':    best_state,
            'best_val_rmse': best_vl,
            'best_hp':       best_hp,
            'n_features':    int(N_FEATURES),
            'fold_idx':      fi + 1,
            'seed_used':     SEED + fi,
            'best_epoch':    best_epoch,
        }, state_path)
        fold_state_paths.append(state_path)

        del model, opt, Xtr_t, ytr_t, Xvl_t
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        print(f'  fold {fi+1}/{N_FOLDS_REFIT}: vl_unit_rmse={best_vl:.6f}  '
              f'epochs={best_epoch}  state→{os.path.basename(state_path)}  '
              f'elapsed={time.time()-t0:.0f}s')

    del Xval_t, Xtest_t
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
else:
    # ── 기존 train_one_fold 흐름 (state 저장 안 함, OOM fallback 활성) ──
    for fi, (tr_uidx, vl_uidx) in enumerate(FOLDS_REFIT):
        X_tr, y_tr_die, X_vl, y_vl_unit_arr, uid_vl, sorted_vl_units, vl_die_mask = _build_fold_args(tr_uidx, vl_uidx)
        result = train_one_fold(
            X_tr, y_tr_die, X_vl, y_vl_unit_arr, uid_vl,
            X_others=[X_val_die_s, X_test_die_s],
            uid_others_list=[uid_val_die, uid_test_die],
            hp=best_hp, verbose=(fi == 0),
            use_compile=USE_COMPILE_REFIT,
        )
        oof_die_arr[vl_die_mask]  = result['oof_die']
        oof_die_mask[vl_die_mask] = True
        val_die_arr  += result['others_die'][0] / N_FOLDS_REFIT
        test_die_arr += result['others_die'][1] / N_FOLDS_REFIT
        fold_records.append({
            'fold':      fi+1,
            'val_rmse':  result['val_rmse'],
            'epochs':    result['epoch'],
        })
        print(f'  fold {fi+1}/{N_FOLDS_REFIT}: vl_unit_rmse={result["val_rmse"]:.6f}  '
              f'epochs={result["epoch"]}  elapsed={time.time()-t0:.0f}s')

assert oof_die_mask.all(), 'OOF die 미커버 fold 있음'
print(f'\n[Refit 완료] {time.time()-t0:.0f}s')

## 8. die→unit 집계 + RMSE

In [ ]:
# die-level → unit-level mean aggregate
oof_unit_arr,  oof_unit_ids  = _aggregate_die_to_unit(oof_die_arr,  uid_train_die)
val_unit_arr,  val_unit_ids  = _aggregate_die_to_unit(val_die_arr,  uid_val_die)
test_unit_arr, test_unit_ids = _aggregate_die_to_unit(test_die_arr, uid_test_die)

# y series 정렬
y_train_aligned = y_train_unit.reindex(oof_unit_ids).values
y_val_aligned   = y_val_unit.reindex(val_unit_ids).values
y_test_aligned  = y_test_unit.reindex(test_unit_ids).values

oof_rmse  = _rmse(oof_unit_arr,  y_train_aligned)
val_rmse  = _rmse(val_unit_arr,  y_val_aligned)
test_rmse = _rmse(test_unit_arr, y_test_aligned)

print('=' * 75)
print(f'  FT-Transformer DIE-LEVEL — best result')
print('=' * 75)
print(f'  {"":12s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"unit RMSE":12s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
# pred 분포 진단 (평균 예측 갇힘 여부)
print(f'  pred std 진단 (평균 예측 탈출 여부):')
print(f'    oof_unit  std = {oof_unit_arr.std():.6f}  (target std = {y_train_aligned.std():.6f})')
print(f'    val_unit  std = {val_unit_arr.std():.6f}  (이전 unit-level NN: 0.000033)')
print(f'    test_unit std = {test_unit_arr.std():.6f}')
import numpy as _np
corr_oof  = float(_np.corrcoef(oof_unit_arr,  y_train_aligned)[0,1])
corr_val  = float(_np.corrcoef(val_unit_arr,  y_val_aligned)[0,1])
corr_test = float(_np.corrcoef(test_unit_arr, y_test_aligned)[0,1])
print(f'  corr(pred, y) (이전 unit-level NN: 0.006):')
print(f'    oof  corr = {corr_oof:.4f}')
print(f'    val  corr = {corr_val:.4f}')
print(f'    test corr = {corr_test:.4f}')
print('-' * 75)
print(f'  비교 기준선:')
print(f'    nn_ft (unit-level — 평균 갇힘): val=0.005822')
print(f'    03b (die-level LGBM):           val=0.005718, test=0.008417')
print(f'    zit_only:                        val=0.005709, test=0.008414')
print(f'    Stacking 11-base (val best):     val=0.005701, test=0.008408')
print('=' * 75)
if val_rmse < 0.0058 and val_unit_arr.std() > 0.001:
    print(f'  ★ 평균 예측 탈출 + plateau 영역 안 → stacking pool 추가 후보')
elif val_unit_arr.std() > 0.001:
    print(f'  → 평균 예측 탈출 OK 이나 plateau 못 들어옴. 잔차 corr 검증 후 결정')
else:
    print(f'  → 여전히 평균 갇힘. die-level 도 NN 한계 — 포기 권장')

## 9. 산출물 저장 (`_temp/nn_ft_die/`)

In [ ]:
def _build_die_df(uid, pred, ddf):
    out = pd.DataFrame({
        KEY_COL:    uid,
        DIE_KEY_COL: ddf[DIE_KEY_COL].values,
        'position': ddf['position'].values,
        'pred':     pred,
    })
    return out

def _build_unit_df(uid_arr, pred_arr, y_series):
    return pd.DataFrame({
        KEY_COL:    uid_arr,
        'pred':     pred_arr,
        'health':   y_series.reindex(uid_arr).values,
    })

# die-level CSV
_build_die_df(uid_train_die, oof_die_arr,  xs_train_die).to_csv(
    os.path.join(OUT_DIR, 'oof_die.csv'), index=False)
_build_die_df(uid_val_die,   val_die_arr,  xs_val_die).to_csv(
    os.path.join(OUT_DIR, 'val_die.csv'), index=False)
_build_die_df(uid_test_die,  test_die_arr, xs_test_die).to_csv(
    os.path.join(OUT_DIR, 'test_die.csv'), index=False)

# unit-level CSV (stacking pool 포맷)
_build_unit_df(oof_unit_ids,  oof_unit_arr,  y_train_unit).to_csv(
    os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(val_unit_ids,  val_unit_arr,  y_val_unit ).to_csv(
    os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(test_unit_ids, test_unit_arr, y_test_unit).to_csv(
    os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

# ── best_params.json (HPO 안 돌렸어도 hp_source 명시) ──
if RUN_HPO and study is not None:
    best_value = study.best_value
    best_params_dict = study.best_trial.params
else:
    best_value = None
    best_params_dict = {k: v for k, v in BEST_HP_FROM_DB.items()
                        if k not in ('n_heads', 'ffn_factor')}

with open(os.path.join(OUT_DIR, 'best_params.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'hp_source':    HP_SOURCE,
        'best_value':   best_value,
        'best_params':  best_params_dict,
        'fixed_params': {'n_heads': 4, 'ffn_factor': 2},
        'fold_records': fold_records,
    }, f, indent=2, ensure_ascii=False, default=str)

# ── HybridScaler pickle (예측 파이프라인 재사용) ──
with open(os.path.join(OUT_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

# ── 환경 / 버전 정보 ──
ENV_INFO = {
    'torch_version':       torch.__version__,
    'cuda_version':        getattr(torch.version, 'cuda', None),
    'cudnn_version':       (torch.backends.cudnn.version()
                            if (DEVICE == 'cuda' and torch.backends.cudnn.is_available())
                            else None),
    'gpu_name':            (torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None),
    'cudnn_benchmark':     bool(torch.backends.cudnn.benchmark),
    'cudnn_deterministic': bool(torch.backends.cudnn.deterministic),
    'tf32_matmul':         bool(torch.backends.cuda.matmul.allow_tf32),
}

if RUN_HPO and study is not None:
    n_pruned   = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
    n_complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
else:
    n_pruned   = 0
    n_complete = 0

meta = {
    'exp_id':            EXP_ID,
    'exp_memo':          EXP_MEMO,
    'model':             'FT-Transformer (die-level + broadcast y + mean agg)',
    'training_level':    'die',
    'target_transform':  'log1p',
    'aggregation':       'mean (die→unit at inference)',
    'n_features':        int(N_FEATURES),
    'n_train_die':       int(X_train_die_s.shape[0]),
    'n_train_unit':      int(len(unit_ids_train_unique)),
    'run_hpo':           RUN_HPO,
    'hp_source':         HP_SOURCE,
    'n_trials':          N_TRIALS if RUN_HPO else 0,
    'n_complete_trials': n_complete,
    'n_pruned_trials':   n_pruned,
    'n_folds_hpo':       N_FOLDS_HPO,
    'n_folds_refit':     N_FOLDS_REFIT,
    'max_epochs':        MAX_EPOCHS,
    'patience':          PATIENCE,
    'device':            DEVICE,
    'use_amp':           USE_AMP,
    'use_compile_hpo':   USE_COMPILE_HPO,
    'use_compile_refit': USE_COMPILE_REFIT,
    'deterministic':     DETERMINISTIC,
    'pruner':            'MedianPruner(n_startup_trials=5, n_warmup_steps=0)',
    'fixed_params':      {'n_heads': 4, 'ffn_factor': 2},
    'optimizer':         'AdamW',
    'scheduler':         None,
    'grad_clip_norm':    1.0,
    'loss_type':         best_hp['loss_type'],
    'scaler':            SCALER_STATS,
    'env':               ENV_INFO,
    'oof_rmse':          oof_rmse,
    'val_rmse':          val_rmse,
    'test_rmse':         test_rmse,
    'pred_std_oof':      float(oof_unit_arr.std()),
    'pred_std_val':      float(val_unit_arr.std()),
    'pred_std_test':     float(test_unit_arr.std()),
    'corr_oof':          float(np.corrcoef(oof_unit_arr,  y_train_aligned)[0,1]),
    'corr_val':          float(np.corrcoef(val_unit_arr,  y_val_aligned)[0,1]),
    'corr_test':         float(np.corrcoef(test_unit_arr, y_test_aligned)[0,1]),
    'preprocess_PARAMS': PARAMS,
    'effective_pp_params': pp['effective_params'],
    'best_params':       best_params_dict,
    'CLIP_Y_EXTREME':    CLIP_Y_EXTREME,
    'feat_cols_clean_n': len(feat_cols_clean),
    'SEED':              int(SEED),
    'save_fold_models':  SAVE_FOLD_MODELS,
    'fold_state_paths':  fold_state_paths,
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f_)
    if os.path.isfile(p):
        sz = os.path.getsize(p) / 1024
        print(f'  {f_:35s}  {sz:>10,.1f} KB')
    else:
        sub_files = os.listdir(p)
        sub_total = sum(os.path.getsize(os.path.join(p, x)) for x in sub_files) / 1024
        print(f'  {f_+"/":35s}  {sub_total:>10,.1f} KB ({len(sub_files)} files)')

# Colab → 로컬 자동 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_base = os.path.join('/content', f'nn_ft_die_{EXP_ID}_outputs')
    _zip_path = shutil.make_archive(_zip_base, 'zip', OUT_DIR)
    print(f'\n[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024:.1f} KB)')
    try:
        files.download(_zip_path)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip_path))
except ImportError:
    pass

## 10. 요약

In [ ]:
print('=' * 75)
print(f' FT-Transformer DIE-LEVEL + HybridScaler — 결과 요약')
print('=' * 75)
print(f'  EXP_ID                : {EXP_ID}')
print(f'  RUN_HPO / hp_source   : {RUN_HPO} / {HP_SOURCE}')
print(f'  training_level        : die (broadcast y, mean agg at inference)')
print(f'  N_FEATURES            : {N_FEATURES} (seq_len {N_FEATURES+1})')
print(f'  train die rows        : {X_train_die_s.shape[0]:,}')
print(f'  device / AMP          : {DEVICE} / {USE_AMP}')
print(f'  compile (HPO/refit)   : {USE_COMPILE_HPO} / {USE_COMPILE_REFIT}')
print(f'  deterministic         : {DETERMINISTIC}')
print(f'  scaler                : HybridScaler (binary={SCALER_STATS["binary_n"]}, '
      f'quantile={SCALER_STATS["quantile_n"]}, power={SCALER_STATS["power_n"]})')

if study is not None:
    print(f'  N_TRIALS / pruned     : {N_TRIALS} / {n_pruned} (complete={n_complete})')
    print(f'  N_FOLDS_HPO/REFIT     : {N_FOLDS_HPO} / {N_FOLDS_REFIT}')
    print(f'  fixed: n_heads=4, ffn_factor=2')
    print(f'  best HP (Optuna):')
    for k, v in study.best_trial.params.items():
        print(f'    {k:15s} = {v}')
else:
    print(f'  N_FOLDS_REFIT         : {N_FOLDS_REFIT} (HPO skipped)')
    print(f'  fixed: n_heads=4, ffn_factor=2')
    print(f'  best HP (from BEST_HP_FROM_DB):')
    for k, v in BEST_HP_FROM_DB.items():
        print(f'    {k:15s} = {v}')

print('-' * 75)
print(f'  {"":10s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"unit RMSE":10s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  진단 (평균 예측 탈출 여부):')
print(f'    pred_std (val)   = {val_unit_arr.std():.6f}  (이전 unit-level NN: 0.000033)')
print(f'    corr(pred, y)val = {np.corrcoef(val_unit_arr, y_val_aligned)[0,1]:.4f}  (이전: -0.0018)')
if SAVE_FOLD_MODELS:
    print(f'  → fold_states/fold_{{1..{N_FOLDS_REFIT}}}.pt 저장됨 (재현/추론 가능)')
print('=' * 75)